# Agile Team Simulator
**Carbon vs Silicon vs Mixed teams — Scrum / Kanban — WIP limit sensitivity**

Models throughput as a function of team composition, methodology, and WIP limits.

Key mechanic:
- Carbon teams degrade non-linearly beyond their optimal WIP range;
- Silicon teams hit a hard throughput ceiling instead of degrading.

In [1]:
import math
import random
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

## 1. Core model

### WIP penalty functions

The central mechanic. Each team type has a different response curve to WIP pressure:

- **Carbon**: bell-curve efficiency — optimal at ~1.5× team size, drops off sharply above 3×
- **Silicon**: flat until a hard API/compute ceiling, then hard cap (no cognitive degradation)
- **Mixed**: weighted blend of both curves

In [2]:
def carbon_wip_efficiency(wip: int, team_size: int = 5) -> float:
    """
    Carbon (human) teams: efficiency peaks around 1.5x team_size WIP,
    then degrades due to context-switching and coordination overhead.
    Returns a multiplier in [0.2, 1.0].

    See Little's law: https://en.wikipedia.org/wiki/Little%27s_law
    on how Work In Progress relates to throughput and lead time
    """

    optimal = 1.5 * team_size

    if wip <= 0:
        return 0.0
    if wip <= optimal:
        # Ramp up to optimal (slight underload penalty)
        return 0.7 + 0.3 * (wip / optimal)
    else:
        # Non-linear degradation above optimal
        #
        # overload_ratio = (wip - optimal) / optimal
        # This normalises how far beyond the optimal WIP you are.
        # If optimal is 7.5 and current WIP is 15, overload_ratio is 1.0: meaning "100% over optimal".
        #
        # penalty = 1.0 - 0.55 * (1 - math.exp(-1.8 * overload_ratio))
        # Penalty includes an exponential decay function. As overload ratio increases it drops from 1 to 0.
        #
        # Maximum penalty is 0.2 because even an overwhelmed team still completes around 20% of work

        overload_ratio = (wip - optimal) / optimal
        penalty = 1.0 - 0.55 * (1 - math.exp(-1.8 * overload_ratio))
        return max(0.2, penalty)

def silicon_wip_efficiency(wip: int, team_size: int = 5, ceiling_multiplier: float = 4.0) -> float:
    """
    Silicon (AI agent) teams: near-flat efficiency up to a hard ceiling
    (rate limits, compute constraints, cost constraints). No cognitive degradation.
    Returns a multiplier in [0.0, 1.0].
    """
    ceiling = ceiling_multiplier * team_size
    if wip <= 0:
        return 0.0
    if wip <= ceiling:
        # Parallelism benefit up to ~half the ceiling, then flat
        ramp = min(wip / (ceiling * 0.5), 1.0)
        return 0.75 + 0.25 * ramp
    else:
        # Hard ceiling: excess WIP just queues, no additional throughput
        return ceiling / wip  # effective throughput normalised


def mixed_wip_efficiency(wip: int, team_size: int = 5,
                         carbon_ratio: float = 0.5) -> float:
    """
    Mixed team: weighted mix. The human fraction limits the ceiling;
    the agent fraction lifts the floor. Sweet spot is 50/50 <-- yes, a guesstimation and based on gut feelings.
    """
    c = carbon_wip_efficiency(wip, team_size)
    s = silicon_wip_efficiency(wip, team_size)

    # Synergy bonus: agents handle overflow, humans handle ambiguity
    synergy = 0.08 * (1 - carbon_ratio) * carbon_ratio  # peaks at 50/50
    return min(1.0, carbon_ratio * c + (1 - carbon_ratio) * s + synergy)

In [3]:
# --- Plot the efficiency curves ---
# TODO: check that the curves are consistent with the model and experiences in the real world

wip_range = list(range(1, 31))
team_size = 5

fig = go.Figure()
fig.add_trace(go.Scatter(x=wip_range,
    y=[carbon_wip_efficiency(w, team_size) for w in wip_range],
    name='Carbon', line=dict(color='#3266ad', width=2)))

fig.add_trace(go.Scatter(x=wip_range,
    y=[silicon_wip_efficiency(w, team_size) for w in wip_range],
    name='Silicon', line=dict(color='#1D9E75', width=2, dash='dash')))

fig.add_trace(go.Scatter(x=wip_range,
    y=[mixed_wip_efficiency(w, team_size) for w in wip_range],
    name='Mixed (50/50)', line=dict(color='#BA7517', width=2, dash='dot')))

fig.add_vline(x=1.5 * team_size, line_dash='dot', line_color='gray',
    annotation_text='Carbon optimum', annotation_position='top right')

fig.update_layout(
    title='WIP efficiency curves (team_size=5)',
    xaxis_title='WIP (items in progress)',
    yaxis_title='Efficiency multiplier',
    yaxis=dict(range=[0, 1.1]),
    height=380,
    template='plotly_white',
    legend=dict(orientation='h', y=-0.2)
)
fig.show()

## 2. Our team and backlog data classes

In [4]:
import dataclasses
from typing import Literal

TeamType = Literal['carbon', 'silicon', 'mixed']
Method = Literal['scrum', 'kanban']

@dataclasses.dataclass
class Team:
    name: str
    team_type: TeamType
    size: int = 5                    # people or agent instances
    wip_limit: int = 8               # explicit WIP limit (Kanban-style)
    carbon_ratio: float = 0.5        # only used for mixed teams

    # Base throughput in items/person/day (before WIP efficiency)
    BASE_RATES = {'carbon': 0.36, 'silicon': 0.80, 'mixed': 0.50}
    # Ceremony overhead fraction of capacity consumed per day
    SCRUM_OVERHEAD = {'carbon': 0.18, 'silicon': 0.04, 'mixed': 0.12}
    KANBAN_OVERHEAD = {'carbon': 0.05, 'silicon': 0.01, 'mixed': 0.04}

    # Retro learning boost per sprint (Scrum only)
    RETRO_BOOST = {'carbon': 0.04, 'silicon': 0.0, 'mixed': 0.02}
    MAX_RETRO_BOOST = {'carbon': 0.30, 'silicon': 0.0, 'mixed': 0.15}

    def __post_init__(self):
        self._retro_accumulated = 0.0

    def apply_retro(self):
        """Call once per sprint end. Accumulates velocity improvement."""
        boost = self.RETRO_BOOST[self.team_type]
        cap = self.MAX_RETRO_BOOST[self.team_type]
        self._retro_accumulated = min(self._retro_accumulated + boost, cap)

    def throughput_today(self, current_wip: int, method: Method) -> float:
        """Items completed today given current WIP and methodology."""
        base = self.BASE_RATES[self.team_type] * self.size
        overhead = (self.SCRUM_OVERHEAD if method == 'scrum'
                    else self.KANBAN_OVERHEAD)[self.team_type]

        if self.team_type == 'carbon':
            wip_eff = carbon_wip_efficiency(current_wip, self.size)
        elif self.team_type == 'silicon':
            wip_eff = silicon_wip_efficiency(current_wip, self.size)
        else:
            wip_eff = mixed_wip_efficiency(current_wip, self.size, self.carbon_ratio)

        return base * wip_eff * (1 - overhead) * (1 + self._retro_accumulated)


@dataclasses.dataclass
class WorkItem:
    item_type: Literal['feature', 'bug', 'story']
    high_priority: bool = False
    complexity: float = 1.0          # effort multiplier (1.0 = 1 story point)
    started_day: int | None = None
    done_day: int | None = None

    @property
    def cycle_time(self) -> int | None:
        if self.started_day is not None and self.done_day is not None:
            return self.done_day - self.started_day
        return None

## 3. Simulation engine

In [5]:
def build_backlog(
    n_features: int = 20,
    n_bugs: int = 15,
    n_stories: int = 25,
    pct_high_priority: float = 0.20,
    seed: int = 42,
) -> list[WorkItem]:
    rng = random.Random(seed)
    items = []
    for _ in range(n_features):
        items.append(WorkItem('feature', complexity=rng.uniform(0.8, 2.0)))
    for _ in range(n_bugs):
        items.append(WorkItem('bug', complexity=rng.uniform(0.3, 1.2)))
    for _ in range(n_stories):
        items.append(WorkItem('story', complexity=rng.uniform(0.5, 1.5)))

    # Mark high-priority items — pull from bugs first, then features
    n_hp = int(len(items) * pct_high_priority)
    bugs = [i for i in items if i.item_type == 'bug']
    rest = [i for i in items if i.item_type != 'bug']
    for item in (bugs + rest)[:n_hp]:
        item.high_priority = True

    # Priority sort: high-priority first, then by complexity (small bugs first)
    items.sort(key=lambda i: (not i.high_priority, i.complexity))
    return items

def run_simulation(
    teams: list[Team],
    backlog: list[WorkItem],
    method: Method = 'scrum',
    sprint_length: int = 10, # Two weeks
    max_days: int = 200,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Runs the simulation and returns a tidy dataframe with one row per (day, team).
    """
    rng = random.Random(seed)
    remaining = [item for item in backlog]  # shallow copy — items are mutable
    in_progress: dict[str, list[WorkItem]] = {t.name: [] for t in teams}
    done_items: list[WorkItem] = []
    records = []

    sprint = 0

    for day in range(1, max_days + 1):
        # --- Sprint boundary (Scrum) ---
        if method == 'scrum' and (day - 1) % sprint_length == 0:
            sprint += 1
            if sprint > 1:
                for team in teams:
                    team.apply_retro()

        # --- Each team pulls and completes work ---
        for team in teams:
            wip = in_progress[team.name]

            # Pull new items up to WIP limit
            while len(wip) < team.wip_limit and remaining:
                item = remaining.pop(0)
                item.started_day = day
                wip.append(item)

            # How much work does this team complete today?
            capacity = team.throughput_today(len(wip), method)

            # Process items (fractional progress tracked via complexity)
            completed_today = 0
            still_wip = []
            for item in wip:
                if capacity >= item.complexity:
                    capacity -= item.complexity
                    item.done_day = day
                    done_items.append(item)
                    completed_today += 1
                else:
                    # Partial progress — reduce complexity for tomorrow
                    item.complexity -= capacity
                    capacity = 0
                    still_wip.append(item)

            in_progress[team.name] = still_wip

            records.append({
                'day': day,
                'sprint': sprint if method == 'scrum' else None,
                'team': team.name,
                'team_type': team.team_type,
                'wip_limit': team.wip_limit,
                'actual_wip': len(wip),
                'completed_today': completed_today,
                'retro_boost': team._retro_accumulated,
                'backlog_remaining': len(remaining),
            })

        if not remaining and all(len(v) == 0 for v in in_progress.values()):
            break  # all done

    df = pd.DataFrame(records)
    df['cumulative_done'] = df.groupby('team')['completed_today'].cumsum()
    df['total_done'] = df.groupby('day')['completed_today'].transform('sum')
    return df

## 4. WIP sensitivity analysis

Run the same backlog across a sweep of WIP limits and compare throughput.

In [6]:
def wip_sensitivity(
    team_type: TeamType,
    wip_limits: list[int],
    method: Method = 'kanban',
    team_size: int = 5,
    seed: int = 42,
) -> pd.DataFrame:
    """Sweep WIP limits for a single team type, return summary stats."""
    results = []

    for wip in wip_limits:
        backlog_template = build_backlog(
            n_features=wip * 3,
            n_bugs=wip * 2,
            n_stories=wip * 3,
            seed=seed
        )

        backlog = [dataclasses.replace(i) for i in backlog_template]  # fresh copy
        team = Team(name=f'{team_type}-wip{wip}', team_type=team_type,
                    size=team_size, wip_limit=wip)
        df = run_simulation([team], backlog, method=method, seed=seed)

        total_done = df['completed_today'].sum()
        days_elapsed = df['day'].max()
        throughput = total_done / days_elapsed if days_elapsed > 0 else 0

        results.append({
            'team_type': team_type,
            'wip_limit': wip,
            'days_to_complete': days_elapsed,
            'throughput_per_day': round(throughput, 3),
            'total_done': int(total_done),
        })

    return pd.DataFrame(results)


# --- Run the sweep ---
wip_range = list(range(1, 21))

carbon_sweep = wip_sensitivity('carbon', wip_range)
silicon_sweep = wip_sensitivity('silicon', wip_range)
mixed_sweep = wip_sensitivity('mixed', wip_range)

sweep_df = pd.concat([carbon_sweep, silicon_sweep, mixed_sweep])
print(sweep_df.groupby('team_type').apply(lambda g: g.loc[g['throughput_per_day'].idxmax()])[
    ['wip_limit', 'throughput_per_day', 'days_to_complete']
].rename(columns={'wip_limit': 'optimal_wip', 'throughput_per_day': 'peak_throughput'}))

           optimal_wip  peak_throughput  days_to_complete
team_type                                                
carbon             7.0            1.556              36.0
mixed              7.0            2.154              26.0
silicon           12.0            3.692              26.0


## 5. Full simulation run with interactive controls

In [7]:
import dataclasses  # already imported above, harmless to repeat

# --- Widget definitions ---
w_method   = widgets.ToggleButtons(options=['scrum', 'kanban'], value='scrum',
                description='Method:', style={'description_width': '60px'})
w_features = widgets.IntSlider(value=20, min=0, max=60, step=5, description='Features')
w_bugs     = widgets.IntSlider(value=15, min=0, max=60, step=5, description='Bugs')
w_stories  = widgets.IntSlider(value=25, min=0, max=60, step=5, description='User stories')
w_priority = widgets.FloatSlider(value=0.20, min=0, max=0.60, step=0.05,
                description='High-prio %', readout_format='.0%')

w_c_teams  = widgets.IntSlider(value=2, min=0, max=6, description='Carbon teams')
w_s_teams  = widgets.IntSlider(value=1, min=0, max=6, description='Silicon teams')
w_m_teams  = widgets.IntSlider(value=1, min=0, max=6, description='Mixed teams')

w_c_wip    = widgets.IntSlider(value=8,  min=1, max=30, description='Carbon WIP')
w_s_wip    = widgets.IntSlider(value=20, min=1, max=40, description='Silicon WIP')
w_m_wip    = widgets.IntSlider(value=12, min=1, max=30, description='Mixed WIP')

btn_run    = widgets.Button(description='▶ Run simulation',
                button_style='primary', layout=widgets.Layout(width='160px'))
out        = widgets.Output()

def on_run(_):
    with out:
        clear_output(wait=True)

        teams = []
        for i in range(w_c_teams.value):
            teams.append(Team(f'Carbon-{i+1}', 'carbon', wip_limit=w_c_wip.value))
        for i in range(w_s_teams.value):
            teams.append(Team(f'Silicon-{i+1}', 'silicon', wip_limit=w_s_wip.value))
        for i in range(w_m_teams.value):
            teams.append(Team(f'Mixed-{i+1}', 'mixed', wip_limit=w_m_wip.value))

        if not teams:
            print('Configure at least one team.')
            return

        backlog = build_backlog(w_features.value, w_bugs.value,
                                w_stories.value, w_priority.value)
        df = run_simulation(teams, backlog, method=w_method.value)

        # --- Cumulative throughput chart ---
        fig = make_subplots(rows=2, cols=2,
                            subplot_titles=[
                                'Cumulative items done per team',
                                'Daily WIP per team',
                                'Throughput (7-day rolling avg)',
                                'Retro velocity boost (Scrum only)'
                            ])

        type_colors = {'carbon': '#3266ad', 'silicon': '#1D9E75', 'mixed': '#BA7517'}
        team_seen = set()

        for team_name, grp in df.groupby('team'):
            tt = grp['team_type'].iloc[0]
            color = type_colors[tt]
            show = team_name not in team_seen
            team_seen.add(team_name)

            fig.add_trace(go.Scatter(
                x=grp['day'], y=grp['cumulative_done'],
                name=team_name, line=dict(color=color, width=1.5),
                legendgroup=team_name, showlegend=show
            ), row=1, col=1)

            fig.add_trace(go.Scatter(
                x=grp['day'], y=grp['actual_wip'],
                name=team_name, line=dict(color=color, width=1.5),
                legendgroup=team_name, showlegend=False
            ), row=1, col=2)

            rolling = grp['completed_today'].rolling(7, min_periods=1).mean()
            fig.add_trace(go.Scatter(
                x=grp['day'], y=rolling.round(2),
                name=team_name, line=dict(color=color, width=1.5),
                legendgroup=team_name, showlegend=False
            ), row=2, col=1)

            fig.add_trace(go.Scatter(
                x=grp['day'], y=(grp['retro_boost'] * 100).round(1),
                name=team_name, line=dict(color=color, width=1.5),
                legendgroup=team_name, showlegend=False
            ), row=2, col=2)

        # Sprint boundary lines (Scrum)
        if w_method.value == 'scrum':
            sprint_days = df[df['sprint'].notna()]['sprint'].unique()
            for s in sprint_days:
                sprint_end_day = df[df['sprint'] == s]['day'].max()
                for r, c in [(1,1),(1,2),(2,1),(2,2)]:
                    fig.add_vline(x=sprint_end_day, line_dash='dot',
                                  line_color='rgba(150,150,150,0.4)', row=r, col=c)

        fig.update_layout(height=680, template='plotly_white',
                          legend=dict(orientation='h', y=-0.1))
        fig.show()

        # --- Summary table ---
        summary = df.groupby(['team', 'team_type', 'wip_limit']).agg(
            days=('day', 'max'),
            total_done=('completed_today', 'sum'),
            avg_wip=('actual_wip', 'mean'),
            peak_retro_boost=('retro_boost', 'max')
        ).reset_index()
        summary['throughput'] = (summary['total_done'] / summary['days']).round(3)
        summary['avg_wip'] = summary['avg_wip'].round(1)
        summary['peak_retro_boost'] = (summary['peak_retro_boost'] * 100).round(1).astype(str) + '%'
        print('\nSummary:')
        display(summary)


btn_run.on_click(on_run)

ui = widgets.VBox([
    widgets.HTML('<b>Methodology</b>'),
    w_method,
    widgets.HTML('<hr><b>Backlog</b>'),
    widgets.HBox([w_features, w_bugs, w_stories]),
    w_priority,
    widgets.HTML('<hr><b>Teams &amp; WIP limits</b>'),
    widgets.HBox([w_c_teams, w_c_wip]),
    widgets.HBox([w_s_teams, w_s_wip]),
    widgets.HBox([w_m_teams, w_m_wip]),
    widgets.HTML('<hr>'),
    btn_run,
    out
])

display(ui)

## 6. Scrum vs Kanban head-to-head

Same team config, same backlog — compare the two methodologies directly.

In [8]:
def compare_methods(
    n_carbon: int = 2, n_silicon: int = 1, n_mixed: int = 1,
    c_wip: int = 8, s_wip: int = 20, m_wip: int = 12,
    **backlog_kwargs
) -> pd.DataFrame:
    backlog_template = build_backlog(**backlog_kwargs)
    rows = []

    for method in ('scrum', 'kanban'):
        teams = ([Team(f'Carbon-{i+1}', 'carbon', wip_limit=c_wip) for i in range(n_carbon)] +
                 [Team(f'Silicon-{i+1}', 'silicon', wip_limit=s_wip) for i in range(n_silicon)] +
                 [Team(f'Mixed-{i+1}', 'mixed', wip_limit=m_wip) for i in range(n_mixed)])

        if not teams:
            continue

        backlog = [dataclasses.replace(i) for i in backlog_template]
        df = run_simulation(teams, backlog, method=method)
        df['method'] = method
        rows.append(df)

    return pd.concat(rows)


cmp = compare_methods(n_features=20, n_bugs=15, n_stories=25)

# Aggregate total throughput per day across all teams
daily_total = (cmp.groupby(['method', 'day'])['completed_today']
               .sum().reset_index(name='done'))
daily_total['cumulative'] = daily_total.groupby('method')['done'].cumsum()

fig = px.line(daily_total, x='day', y='cumulative', color='method',
              color_discrete_map={'scrum': '#3266ad', 'kanban': '#1D9E75'},
              title='Scrum vs Kanban — cumulative throughput (all teams combined)',
              labels={'cumulative': 'items done', 'day': 'day'},
              template='plotly_white', height=380)
fig.show()